In [2]:
# 환경변수 / 임베딩 / 분할 / 벡터스토어 / 로더 준비
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader

load_dotenv()

/var/folders/s0/pp12ryd171g85gths6jn_18c0000gn/T/ipykernel_5793/3870073172.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


True

In [5]:
# 1. 원본 문서를 Chunk로 분할
# chunk_size가 너무 크면 검색 정밀도가 떨어질 수 있고,
# 너무 작으면 문맥이 잘릴 수 있으므로 실제 데이터로 실험이 필요하다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=0
)

loader1 = TextLoader("data/nlp-keywords.txt")
loader2 = TextLoader("data/finance-keywords.txt")

# load_and_split(): 문서 로드 + Text Splitter 적용
split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

len(split_doc1), len(split_doc2)

(11, 6)

In [ ]:
# 2. Chunk를 임베딩하여 Chroma Vector Store 생성
# Chroma.from_documents()가 각 Document의 page_content를 임베딩하고 저장한다.
db = Chroma.from_documents(
    documents=split_doc1,
    embedding=OpenAIEmbeddings(),
    collection_name="my_db"  # 같은 목적의 벡터들을 묶는 논리적 그룹
)

In [ ]:
DB_PATH = "./chroma_db"  # Chroma 데이터를 로컬 디스크에 저장할 경로

# persist_directory를 지정하면 프로그램 종료 후에도 데이터가 유지된다.
# 지정하지 않으면 일반적으로 현재 실행 환경에서 임시로 사용한다.
persist_db = Chroma.from_documents(
    documents=split_doc1,
    embedding=OpenAIEmbeddings(),
    persist_directory=DB_PATH,
    collection_name="my_db"
)

In [ ]:
# 이미 디스크에 저장된 Chroma Collection 다시 연결
# 같은 persist_directory / collection_name / embedding model을 사용한다.
persist_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=OpenAIEmbeddings(),
    collection_name="my_db"
)

In [9]:
# 저장된 id / document / metadata 등을 조회
# 유사도 검색이 아니라 Vector Store 내부 데이터를 직접 확인하는 용도
persist_db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [10]:
# Document 객체를 직접 만들지 않고 문자열 목록으로 Vector Store 생성
db2 = Chroma.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding=OpenAIEmbeddings(),
)

In [11]:
# from_texts()로 저장된 데이터 확인
db2.get()

{'ids': ['85a8ef16-50ea-43ac-8c1b-675ceb6a29ad',
  '3731273c-efc6-48fb-b478-b71455fdc7f8'],
 'embeddings': None,
 'documents': ['안녕하세요. 정말 반갑습니다.', '제 이름은 테디입니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None, None]}

In [12]:
# 질문도 같은 임베딩 모델로 벡터화한 뒤,
# 저장된 문서 벡터들과 비교해 의미적으로 가까운 문서를 반환한다.
db.similarity_search("TF IDF에 대하여 알려줘")

[Document(id='f53146f4-a3ba-4559-852a-2a08f186e0b1', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='39a42078-d3f6-4f33-a464-d9db0a77a908', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이

In [13]:
# k: 유사도가 높은 문서를 몇 개까지 가져올지 지정
db.similarity_search("TF IDF에 대하여 알려줘", k=2)

[Document(id='f53146f4-a3ba-4559-852a-2a08f186e0b1', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='39a42078-d3f6-4f33-a464-d9db0a77a908', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이

In [ ]:
# metadata filter를 적용하면 특정 조건의 문서 안에서만 검색한다.
db.similarity_search(
    "TF IDF에 대하여 알려줘",
    filter={"source": "data/finance-keywords.txt"},
    k=2
)

# finance 문서 안에 질문과 관련된 내용이 없다면 적절한 결과가 나오지 않을 수 있다.

[]

In [ ]:
from langchain_core.documents import Document

# Vector Store에 새 Document 추가
# page_content는 임베딩 대상, metadata는 필터/출처 표시 등에 활용한다.
db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼게요",
            metadata={"source": "mydata.txt"},
            # id를 직접 지정하면 이후 조회/삭제/업데이트 시 사용하기 편하다.
            id="1",
        )
    ]
)

['1']

In [16]:
# id가 "1"인 데이터 조회
db.get("1")

{'ids': ['1'],
 'embeddings': None,
 'documents': ['안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼게요'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'mydata.txt'}]}

In [17]:
# 문자열을 바로 추가하는 add_texts()
# 같은 id를 사용하면 Vector Store 구현에 따라 기존 항목을 갱신하는 식으로 사용할 수 있다.
db.add_texts(
    ["이전에 추가한 Document를 덮어쓰겠습니다.", "덮어쓴 결과가 어떤가요?"],
    metadatas=[
        {"source": "mydata.txt"},
        {"source": "mydata.txt"}
    ],
    ids=["1", "2"],
)

['1', '2']

In [18]:
# id=1 데이터가 어떻게 저장되었는지 확인
db.get(["1"])

{'ids': ['1'],
 'embeddings': None,
 'documents': ['이전에 추가한 Document를 덮어쓰겠습니다.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'mydata.txt'}]}

In [19]:
# id 기준 삭제
db.delete(ids=["1"])

# 삭제 여부 확인
db.get(["1", "2"])

{'ids': ['2'],
 'embeddings': None,
 'documents': ['덮어쓴 결과가 어떤가요?'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'mydata.txt'}]}

In [ ]:
# where: metadata 조건으로 데이터 자체를 조회
# similarity_search의 filter와 목적은 비슷하지만 get()은 저장 데이터 조회에 가깝다.
db.get(
    where={"source": "mydata.txt"}
)

{'ids': ['2'],
 'embeddings': None,
 'documents': ['덮어쓴 결과가 어떤가요?'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'mydata.txt'}]}

In [ ]:
# Collection: Vector Store 내부에서 데이터를 묶어 관리하는 논리적 그룹
# reset_collection()은 현재 collection의 데이터를 초기화하므로 사용 시 주의
db.reset_collection()
db.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [22]:
# NLP + 금융 문서를 하나의 Collection에 저장
# 이후 Retriever가 이 Collection 전체를 대상으로 검색한다.
db = Chroma.from_documents(
    documents=split_doc1 + split_doc2,
    embedding=OpenAIEmbeddings(),
    collection_name="nlp",
)

In [ ]:
# 3. Vector Store를 Retriever 인터페이스로 변환
# Retriever는 "질문을 받아 관련 Document를 반환"하는 공통 검색 인터페이스이다.
retriever = db.as_retriever()

# 내부적으로 질문 임베딩 → Vector Store 유사도 검색 → 관련 Document 반환
retriever.invoke("Word2Vec에 대하여 알려줘")

[Document(id='7c5715a7-9483-4c0d-965c-56c08e3c4e7f', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='bedd155b-f80c-4652-b744-bfc54476cb23', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.\n예시: HuggingFace의 Transformers 라이브러리를 사용하여 감정 분석, 텍스트 생성 등의 작업을 수행할 수 있

In [ ]:
# MMR(Maximal Marginal Relevance):
# 질문과의 관련성뿐 아니라 검색 결과끼리 너무 비슷하지 않도록 다양성도 고려한다.
#
# k           : 최종 반환 문서 수
# fetch_k     : MMR이 후보로 먼저 가져올 문서 수
# lambda_mult : 관련성 vs 다양성 비율 (낮을수록 다양성 쪽 비중 증가)
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "lambda_mult": 0.25,
        "fetch_k": 10
    }
)

retriever.invoke("Word2Vec에 대하여 알려줘")

[Document(id='7c5715a7-9483-4c0d-965c-56c08e3c4e7f', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='2a83e166-dea5-46dc-abfb-a58e54f3c728', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\

In [ ]:
# similarity_score_threshold:
# 상위 k개를 무조건 가져오기보다 일정 유사도 이상인 문서만 반환
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.8}
)

retriever.invoke("Word2Vec에 대하여 알려줘")

[Document(id='7c5715a7-9483-4c0d-965c-56c08e3c4e7f', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source')]

In [ ]:
# Retriever에서도 metadata filter를 적용할 수 있다.
# 여기서는 finance-keywords.txt에서만 ESG 관련 문서를 검색한다.
retriever = db.as_retriever(
    search_kwargs={
        "filter": {"source": "data/finance-keywords.txt"},
        "k": 2
    }
)

retriever.invoke("ESG에 대하여 알려줘")

[Document(id='8fdf38bb-4f57-4617-aeb8-5df4b20b4b6b', metadata={'source': 'data/finance-keywords.txt'}, page_content='정의: ESG는 기업의 환경, 사회, 지배구조 측면을 고려하는 투자 접근 방식입니다.\n예시: S&P 500 ESG 지수는 우수한 ESG 성과를 보이는 기업들로 구성된 지수입니다.\n연관키워드: 지속가능 투자, 기업의 사회적 책임, 윤리 경영\n\nStock Buyback\n\n정의: 자사주 매입은 기업이 자사의 주식을 시장에서 다시 사들이는 것을 말합니다.\n예시: 애플은 S&P 500 기업 중 가장 큰 규모의 자사주 매입 프로그램을 운영하고 있습니다.\n연관키워드: 주주 가치, 자본 관리, 주가 부양\n\nCyclical Stocks\n\n정의: 경기순환주는 경제 상황에 따라 실적이 크게 변동하는 기업의 주식을 말합니다.\n예시: 포드, 제너럴 모터스와 같은 자동차 기업들은 S&P 500에 포함된 대표적인 경기순환주입니다.\n연관키워드: 경제 사이클, 섹터 분석, 투자 타이밍\n\nDefensive Stocks\n\n정의: 방어주는 경기 변동에 상관없이 안정적인 실적을 보이는 기업의 주식을 의미합니다.\n예시: 프록터앤갬블, 존슨앤존슨과 같은 생활필수품 기업들은 S&P 500 내 대표적인 방어주로 꼽힙니다.\n연관키워드: 안정적 수익, 저변동성, 리스크 관리'),
 Document(id='2c16bbc7-b252-421c-b7bf-3e21794849b0', metadata={'source': 'data/finance-keywords.txt'}, page_content='정의: 주식 리서치는 기업의 재무 상태, 사업 모델, 경쟁력 등을 분석하여 투자 의사 결정을 돕는 활동입니다.\n예시: 골드만삭스의 애널리스트들이 S&P 500 기업들에 대한 분기별 실적 전망을 발표했습니다.\n연관키워드: 투자 분석, 기업 가치평가, 시장 전망\n\nCorporate 